In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#15.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");
//var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble3D");

Opening existing database 'P:\hpccluster\RisingBubble2D'.


In [ ]:
var sessions = db.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_MismatchFix_restart3*	09/04/2024 14:34:52	a7ac51e0...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_restart3*	09/04/2024 12:53:16	d304f983...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart3*	09/02/2024 11:49:28	60901f69...
#3: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	08/30/2024 09:22:51	44e79500...
#4: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	08/30/2024 09:35:54	46f02668...
#5: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#6: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#7: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


In [ ]:
var sess = sessions.Pick(5);
sess

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...

In [ ]:
var tStep_fail = sess.Timesteps.Last();
tStep_fail

 { Time-step: 1102; Physical time: 2.201999999999979s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

In [ ]:
var tStep_OK = sess.Timesteps.Pick(sess.Timesteps.Count()-2);
tStep_OK

 { Time-step: 1101; Physical time: 2.201999999999979s; Fields: Phi, PhiDG, VelocityX, VelocityY, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, GravityY#A, GravityY#B, VelocityX@Phi, VelocityY@Phi, MPIrank, CutCells; Name:  }

## Setting up for some checks

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
//var tStep = tStep_fail;
var tStep = tStep_OK;

In [ ]:
SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
GridData grdData = (GridData)phi.GridDat; 
LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
levelSet.Acc(1.0, phi); 
LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
LsTrk.UpdateTracker(tStep.PhysicalTime);

In [ ]:
CellMask CCmask = LsTrk.Regions.GetCutCellMask();
CCmask.ItemEnum.Count()

40

In [ ]:
int order = 2 * phi.Basis.Degree + 1;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

In [ ]:
SpeciesId spcId_A = LsTrk.GetSpeciesId("A");
SpeciesId spcId_B = LsTrk.GetSpeciesId("B");

## Checking Integral properties

In [ ]:
using BoSSS.Foundation.XDG.Quadrature.HMF;

In [ ]:
//DivergenceFreeBasis DivFreeBasis = new DivergenceFreeBasis(LsTrk.GridDat, LsTrk.GridDat.Cells.RefElements[0], order + 1);
Basis ScalarBasis = new Basis(LsTrk.GridDat, 0); 

In [ ]:
SinglePhaseField testFieldX = new SinglePhaseField(ScalarBasis);
Func<double[], double> fx = (X => 1.0);
testFieldX.ProjectField(fx);
SinglePhaseField testFieldY = new SinglePhaseField(ScalarBasis);
Func<double[], double> fy = (X => 1.0);
testFieldY.ProjectField(fy);
SinglePhaseField testFieldZ = new SinglePhaseField(ScalarBasis);
Func<double[], double> fz = (X => 1.0);
testFieldZ.ProjectField(fz);

In [ ]:
VectorField<SinglePhaseField> testField = new VectorField<SinglePhaseField>(new SinglePhaseField[] {testFieldX, testFieldY});

In [ ]:
double gaussOnDomain = 0.0;
CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SchemeHelper.GetLevelSetquadScheme(0, CCmask).Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        int qN = QR.NoOfNodes;
        int D = LsTrk.GridDat.SpatialDimension;

        var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

        for (int d = 0; d < D; d++) {

            var fEval_d = MultidimensionalArray.Create(length, qN);
            testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

            for (int i = 0; i < length; i++) {
                for (int qn = 0; qn < qN; qn++) {
                    EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                }
            }
        }

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            gaussOnDomain += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
gaussOnDomain

6.072578118826444E-08

In [ ]:
public static (double error, double gaussInVolume, double gaussAtInterface, double gaussAtEdges) CheckGaussForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double gaussInVolume = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fGrad_d = MultidimensionalArray.Create(length, qN, D);
                testField[d].EvaluateGradient(i0, length, QR.Nodes, fGrad_d, 0, 0.0);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fGrad_d[i, qn, d];
                    }
                }
            }
            //EvalResult.SetAll(1.0);
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussInVolume += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double gaussAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);        // TODO change direction for spcId_B

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEval_d[i, qn] * LsNormals[i, qn, d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);
            
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                gaussAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);

    double gaussAtCutEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetEdgeQuadScheme(spcId_A, IntegrationDomain: edgeIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            for (int d = 0; d < D; d++) {

                var fEvalEdgeIN_d = MultidimensionalArray.Create(length, qN);
                var fEvalEdgeOUT_d = MultidimensionalArray.Create(length, qN);
                testField[d].EvaluateEdge(i0, length, QR.Nodes, fEvalEdgeIN_d, fEvalEdgeOUT_d);

                for (int i = 0; i < length; i++) {
                    double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();  
                    if (LsTrk.GridDat.Edges.CellIndices[i0 + i, 1] == jCell) {
                        // Console.WriteLine($"jCell {jCell} OUT on edge {i0 + i} - change direction");
                        EdgeNormal.ScaleV(-1.0);
                    }
                    // Console.WriteLine($"Edge normal: ({EdgeNormal[0]}, {EdgeNormal[1]})");

                    for (int qn = 0; qn < qN; qn++) {
                        EvalResult[i, qn, 0] += fEvalEdgeIN_d[i, qn] * EdgeNormal[d];
                    }
                }
            }
            // EvalResult.SetAll(1.0);

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                // Console.WriteLine($"edge {i0 + i}");
                gaussAtCutEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = gaussInVolume - gaussAtInterface - gaussAtCutEdges;

    return (error, gaussInVolume, gaussAtInterface, gaussAtCutEdges);

}

In [ ]:
int jCell = LsTrk.Regions.GetCutCellMask().ItemEnum.First();
jCell

135

In [ ]:
CheckGaussForCell(jCell, LsTrk, spcId_A, testField, order)

Item1,Item2,Item3,Item4
7.750634439807058E-10,0,-0.03526752791947972,0.03526752714441628


In [ ]:
public static (double error, double stokesAtInterface, double stokesAtEdges) CheckStokesForCell(int jCell, LevelSetTracker LsTrk, SpeciesId spcId_A, VectorField<SinglePhaseField> testField, int order) {

    GridData grdDat = LsTrk.GridDat;
    BitArray cellIntDomBA = new BitArray(grdDat.Cells.NoOfLocalUpdatedCells);
    cellIntDomBA[jCell] = true;
    CellMask cellIntDom = new CellMask(grdDat, cellIntDomBA); //, MaskType.Geometrical);

    XQuadSchemeHelper schemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;

    // ========================================================
    double stokesAtInterface = 0.0;
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        schemeHelper.GetLevelSetquadScheme(0, cellIntDom).Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;

            var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(QR.Nodes, i0, length);

            var curv = MultidimensionalArray.Create(length, qN);                              
            ((LevelSet)LsTrk.LevelSets[0]).EvaluateTotalCurvature(i0, length, QR.Nodes, curv);

            for (int d = 0; d < D; d++) {

                var fEval_d = MultidimensionalArray.Create(length, qN);
                testField[d].Evaluate(i0, length, QR.Nodes, fEval_d);

                for (int i = 0; i < length; i++) {
                    for (int qn = 0; qn < qN; qn++) {  
                        EvalResult[i, qn, 0] += curv[i, qn] * LsNormals[i, qn, d] * fEval_d[i, qn];
                    }
                }
            }

        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtInterface += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();
    
    // =========================================================
    BitArray edgeIntDomBA = new BitArray(grdDat.iGeomEdges.Count);
    var jCell2Edges = grdDat.Cells.Cells2Edges[jCell];
    foreach (int iE in jCell2Edges) {
        if (iE < 0) 
            edgeIntDomBA[(-iE) - 1] = true;
        else 
            edgeIntDomBA[iE - 1] = true;
    }
    EdgeMask edgeIntDom = new EdgeMask(grdDat, edgeIntDomBA);
    
    var testFactory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
    EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(testFactory, edgeIntDom);
    
    double stokesAtEdges = 0.0;
    EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
        delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {
    
            int qN = QR.NoOfNodes;
            int D = LsTrk.GridDat.SpatialDimension;
    
            for (int i = 0; i < length; i++) {
    
                double[] EdgeNormal = LsTrk.GridDat.Edges.NormalsForAffine.ExtractSubArrayShallow(i0 + i, -1).To1DArray();
                // Console.WriteLine($"Edge normal: ({EdgeNormal[0]}, {EdgeNormal[1]})");  
                    
                // Console.WriteLine($"edge {i0 + i} - number of nodes {qN}");
                int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0 + i, 0];
                NodeSet VolNodes = QR.Nodes.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
                // Console.WriteLine($"volume node: ({VolNodes[0, 0]}, {VolNodes[0, 1]})");  
        
                var LsNormals = LsTrk.DataHistories[0].Current.GetLevelSetNormals(VolNodes, jCell, length);
                    
                for (int d = 0; d < D; d++) {
    
                    var fEval = MultidimensionalArray.Create(length, qN);
                    testField[d].Evaluate(jCell, length, VolNodes, fEval);
    
                    for (int qn = 0; qn < qN; qn++) {
    
                        if (D != 2) 
                            throw new ArgumentException();
    
                        double[] LsTangent = new double[2];
                        
                        LsTangent[0] = -LsNormals[i, qn, 1];
                        LsTangent[1] = LsNormals[i, qn, 0];
            
                        if (GenericBlas.InnerProd(LsTangent, EdgeNormal) < 0.0)
                            LsTangent.ScaleV(-1.0);
    
                        // Console.WriteLine($"LS tangent: ({LsTangent[0]}, {LsTangent[1]})");  
                        EvalResult[i, qn, 0] += LsTangent[d] * fEval[i, qn];
                    }
                }
            }
    
        },
        delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
            for(int i = 0; i < length; i++) {
                stokesAtEdges += ResultsOfIntegration[i, 0];
            }
        }
    ).Execute();

    // =========================================================
    double error = stokesAtInterface + stokesAtEdges;

    return (error, stokesAtInterface, stokesAtEdges);
}

In [ ]:
CheckStokesForCell(jCell, LsTrk, spcId_A, testField, order)

Item1,Item2,Item3
-2.1448593483242018E-06,-0.2001008457938563,0.200098700934508


## Evaluate Interface Continuity

In [ ]:
int order = phi.Basis.Degree;
XQuadSchemeHelper SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadSchemeHelper;
EdgeQuadratureScheme SurfaceElement_Edge = SchemeHelper.Get_SurfaceElement_EdgeQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
//SurfaceElement_Edge.Domain.ItemEnum.Count()

In [ ]:
EdgeMask CutCellBoundaryEdgeMask = LsTrk.Regions.GetCutCellMask().AllEdges().Except(LsTrk.Regions.GetCutCellMask().GetAllInnerEdgesMask());
//CutCellBoundaryEdgeMask.ItemEnum.Count()

In [ ]:
var factory = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order).XQuadFactoryHelper.GetSurfaceElement_BoundaryRuleFactory(0, LsTrk.GridDat.Grid.RefElements[0]);
EdgeQuadratureScheme SurfaceElement_BoundaryEdge = new EdgeQuadratureScheme(factory, CutCellBoundaryEdgeMask);

In [ ]:
double result = 0;
int D = LsTrk.GridDat.SpatialDimension;
EdgeQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
    SurfaceElement_BoundaryEdge.Compile(LsTrk.GridDat, order),
    delegate (int i0, int length, QuadRule QR, MultidimensionalArray EvalResult) {

        // inner quadrature node
        NodeSet Enode_l = QR.Nodes;
        int trf = LsTrk.GridDat.Edges.Edge2CellTrafoIndex[i0, 0];
        NodeSet Vnode_l = Enode_l.GetVolumeNodeSet(LsTrk.GridDat, trf, false);
        NodeSet Vnode_g = Vnode_l.CloneAs();
        int cell = LsTrk.GridDat.Edges.CellIndices[i0, 0];
        LsTrk.GridDat.TransformLocal2Global(Vnode_l, Vnode_g, cell);
        if (D == 2)
            Console.WriteLine("inner quadrature node: ({0},{1})", Vnode_g[0, 0], Vnode_g[0, 1]);
        else 
            Console.WriteLine("inner quadrature node: ({0},{1},{2})", Vnode_g[0, 0], Vnode_g[0, 1], Vnode_g[0, 2]);

        // for(int i = 0; i < length; i++) {
        //     EvalResult[i, 0, 0] = 1;    
        // }
        EvalResult.SetAll(1.0);

    },
    delegate (int i0, int length, MultidimensionalArray ResultsOfIntegration) {
        for(int i = 0; i < length; i++) {
            result += ResultsOfIntegration[i, 0];
        }
    }
).Execute();
result

0

In [ ]:
CellQuadratureScheme ContactLineVolumeScheme = SchemeHelper.GetContactLineQuadScheme(LsTrk.GetSpeciesId("A"), 0);

In [ ]:
ContactLineVolumeScheme.Domain.ItemEnum.Count()

0

## Continuity projection

In [ ]:
using BoSSS.Solution.LevelSetTools;

In [ ]:
SinglePhaseField phiDG = (SinglePhaseField)tStep.GetField("PhiDG");

In [ ]:
ContinuityProjection preEnforcer = new ContinuityProjection(
    phi.Basis,
    phiDG.Basis,
    LsTrk.GridDat,
    ContinuityProjectionOption.ConstrainedDG);